# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

The baseline ranks eligible client-content pairs by a transparent estimate of future impression-decline risk. It uses February features only; March fields are used only after ranking to evaluate the result.

The score uses these reason codes:

- `high_visibility_at_risk`: at least 1,000 prior impressions and a weak prior position.
- `weak_position_signal`: prior average position is worse than 10.
- `low_prior_engagement`: prior engagement rate is below 30% when sessions are available.
- `low_click_through_rate`: prior clicks are low relative to prior impressions.
- `limited_prior_visibility`: fewer than 1,000 prior impressions, but still at least 100 and eligible.

A page can receive more than one signal. The score is a prioritization baseline, not a causal claim.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

# Load the prepared local dataset created in Week 2.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)

# Evaluate only pages with enough prior impressions for a stable decline label.
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()

# Derive one February CTR feature. It uses no March information.
dataframe['prior_ctr'] = (
    dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)
)

# Transparent binary signals from the February feature window.
dataframe['high_visibility_at_risk'] = (
    (dataframe['prior_impressions'] >= 1000)
    & (dataframe['prior_avg_position'] > 10)
).astype(int)
dataframe['weak_position_signal'] = (
    dataframe['prior_avg_position'] > 10
).fillna(False).astype(int)
dataframe['low_prior_engagement'] = (
    (dataframe['prior_sessions'] > 0)
    & (dataframe['prior_engagement_rate'] < 0.30)
).fillna(False).astype(int)
dataframe['low_click_through_rate'] = (
    dataframe['prior_ctr'] < 0.01
).fillna(False).astype(int)
dataframe['limited_prior_visibility'] = (
    dataframe['prior_impressions'] < 1000
).astype(int)

# Low-volume eligible pages are more vulnerable to large percentage changes.
# The other signals add risk points without using the March outcome.
dataframe['baseline_score'] = (
    3 * dataframe['limited_prior_visibility']
    + dataframe['weak_position_signal']
    + dataframe['low_prior_engagement']
    + dataframe['low_click_through_rate']
)

def make_reason_code(row):
    reasons = []
    if row['limited_prior_visibility']:
        reasons.append('limited_prior_visibility')
    if row['weak_position_signal']:
        reasons.append('weak_position_signal')
    if row['low_prior_engagement']:
        reasons.append('low_prior_engagement')
    if row['low_click_through_rate']:
        reasons.append('low_click_through_rate')
    if not reasons:
        reasons.append('higher_prior_visibility')
    return '; '.join(reasons)

dataframe['reason_code'] = dataframe.apply(make_reason_code, axis=1)

dataframe[['client_hash_id', 'content_hash_id', 'baseline_score', 'reason_code']].head()

,client_hash_id,content_hash_id,baseline_score,reason_code
45,client_e547b89c05043229,content_8f1d39fd7d22589e,6,limited_prior_visibility; weak_position_signal...
46,client_e547b89c05043229,content_560bb5e90548712d,4,limited_prior_visibility; low_click_through_rate
47,client_e547b89c05043229,content_a788d093ce1ae235,3,weak_position_signal; low_prior_engagement; lo...
48,client_e547b89c05043229,content_8ebb700097667e14,3,weak_position_signal; low_prior_engagement; lo...
49,client_e547b89c05043229,content_74e44486883ee085,5,limited_prior_visibility; weak_position_signal...


## 2. Build the ranked queue and evaluate it

The queue uses the February-only baseline score. It is then evaluated against the observed March decline label. The primary metric is precision@20: the share of the top 20 ranked rows that actually declined in March.

The output is written to `work/outputs/february_march_baseline_queue.csv`. This generated CSV is ignored by Git.

In [4]:
# Rank the eligible pages using the February-only baseline score.
ranked = dataframe.sort_values(
    ['baseline_score', 'prior_impressions'],
    ascending=[False, True],
).reset_index(drop=True)
ranked['rank'] = np.arange(1, len(ranked) + 1)

output_path = repo_root / 'work' / 'outputs' / 'february_march_baseline_queue.csv'
ranked_columns = [
    'rank',
    'client_hash_id',
    'content_hash_id',
    'prior_impressions',
    'prior_clicks',
    'prior_ctr',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
    'future_impressions',
    'future_decline_label',
    'baseline_score',
    'reason_code',
]
ranked[ranked_columns].to_csv(output_path, index=False)

# Precision@20 is the mean target value in the first 20 ranked rows.
top_20 = ranked.head(20)
precision_at_20 = top_20['future_decline_label'].mean()
base_rate = ranked['future_decline_label'].mean()

metrics = pd.DataFrame({
    'metric': ['eligible_rows', 'future_decline_base_rate', 'precision_at_20'],
    'value': [len(ranked), base_rate, precision_at_20],
})

print(f"Saved queue to: {output_path}")
print(metrics)
ranked[ranked_columns].head(10)

Saved queue to: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\february_march_baseline_queue.csv
                     metric         value
0             eligible_rows  80322.000000
1  future_decline_base_rate      0.217549
2           precision_at_20      0.200000


,rank,client_hash_id,content_hash_id,prior_impressions,prior_clicks,prior_ctr,prior_avg_position,prior_sessions,prior_engagement_rate,future_impressions,future_decline_label,baseline_score,reason_code
0,1,client_23a62021009f63c4,content_fb61ce17a12b8a8c,100.0,0.0,0.000000,20.690000,6.0,0.000000,88.0,0,6,limited_prior_visibility; weak_position_signal...
1,2,client_e547b89c05043229,content_2a896bf67c6373e6,100.0,0.0,0.000000,19.840000,1.0,0.000000,127.0,0,6,limited_prior_visibility; weak_position_signal...
2,3,client_23a62021009f63c4,content_1b310b4ea927bd9a,100.0,0.0,0.000000,14.550000,1.0,0.000000,61.0,1,6,limited_prior_visibility; weak_position_signal...
3,4,client_ff644d8251367cbb,content_c16c2366a117538c,100.0,0.0,0.000000,54.320000,3.0,0.000000,128.0,0,6,limited_prior_visibility; weak_position_signal...
4,5,client_3197e6291363b4db,content_73c40d8399d316bd,100.0,0.0,0.000000,51.510000,1.0,0.000000,7.0,1,6,limited_prior_visibility; weak_position_signal...
5,6,client_23a62021009f63c4,content_e6fdbd4f7a34e475,100.0,0.0,0.000000,12.370000,1.0,0.000000,42.0,1,6,limited_prior_visibility; weak_position_signal...
6,7,client_3197e6291363b4db,content_2f862db8321adb52,100.0,0.0,0.000000,24.310000,1.0,0.000000,92.0,0,6,limited_prior_visibility; weak_position_signal...
7,8,client_b10cb2997d0c7c86,content_b5e7a2080c044f46,101.0,0.0,0.000000,45.970297,1.0,0.000000,16.0,1,6,limited_prior_visibility; weak_position_signal...
8,9,client_9958f0a7ae1df715,content_56da6a86f01529c2,101.0,0.0,0.000000,23.554455,6.0,0.166667,200.0,0,6,limited_prior_visibility; weak_position_signal...
9,10,client_65de48885f4ef01b,content_eff2a77134c1f048,101.0,1.0,0.009901,11.128713,4.0,0.000000,83.0,0,6,limited_prior_visibility; weak_position_signal...


## 3. Top-20 review

The top 20 is the practical review queue. For each row, I retain the client and content context, the February evidence, the reason code, and the observed March label for evaluation.

The baseline does not claim that these pages will definitely decline or that the score caused anything. It only tests whether a simple, explainable ranking can concentrate future declines near the top.

In [5]:
review_top_20 = ranked.head(20).copy()
review_top_20['action'] = 'review_for_possible_impression_decline'
review_top_20['confidence_note'] = np.where(
    review_top_20['baseline_score'] >= 6,
    'higher baseline signal strength',
    'moderate baseline signal strength',
)
review_top_20['what_would_make_it_wrong'] = (
    'seasonality, tracking gaps, consolidation, or a short-lived fluctuation'
)

review_columns = [
    'rank',
    'client_hash_id',
    'content_hash_id',
    'baseline_score',
    'reason_code',
    'action',
    'confidence_note',
    'future_decline_label',
    'what_would_make_it_wrong',
]
review_top_20[review_columns]

,rank,client_hash_id,content_hash_id,baseline_score,reason_code,action,confidence_note,future_decline_label,what_would_make_it_wrong
0,1,client_23a62021009f63c4,content_fb61ce17a12b8a8c,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,0,"seasonality, tracking gaps, consolidation, or ..."
1,2,client_e547b89c05043229,content_2a896bf67c6373e6,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,0,"seasonality, tracking gaps, consolidation, or ..."
2,3,client_23a62021009f63c4,content_1b310b4ea927bd9a,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,1,"seasonality, tracking gaps, consolidation, or ..."
3,4,client_ff644d8251367cbb,content_c16c2366a117538c,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,0,"seasonality, tracking gaps, consolidation, or ..."
4,5,client_3197e6291363b4db,content_73c40d8399d316bd,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,1,"seasonality, tracking gaps, consolidation, or ..."
5,6,client_23a62021009f63c4,content_e6fdbd4f7a34e475,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,1,"seasonality, tracking gaps, consolidation, or ..."
6,7,client_3197e6291363b4db,content_2f862db8321adb52,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,0,"seasonality, tracking gaps, consolidation, or ..."
7,8,client_b10cb2997d0c7c86,content_b5e7a2080c044f46,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,1,"seasonality, tracking gaps, consolidation, or ..."
8,9,client_9958f0a7ae1df715,content_56da6a86f01529c2,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,0,"seasonality, tracking gaps, consolidation, or ..."
9,10,client_65de48885f4ef01b,content_eff2a77134c1f048,6,limited_prior_visibility; weak_position_signal...,review_for_possible_impression_decline,higher baseline signal strength,0,"seasonality, tracking gaps, consolidation, or ..."


## 4. Weak picks and leakage check

The score must use only February columns. March columns are retained only for evaluation after ranking. I will also check that every eligible row has a label, that the queue has no duplicate client-content pairs, and that the top 20 is not empty.

The strongest alternative explanations are seasonality, tracking gaps, consolidation between related pages, and random variation. A reviewer should consider these before taking action.

In [6]:
# Confirm the baseline uses only February-derived columns.
feature_columns = [
    'prior_impressions',
    'prior_clicks',
    'prior_ctr',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
]
future_columns = ['future_impressions', 'future_decline_label']

pair_duplicates = ranked.duplicated(
    subset=['client_hash_id', 'content_hash_id']
).sum()

leakage_check = pd.DataFrame({
    'check': [
        'feature_columns_are_prior_period',
        'future_columns_used_only_for_evaluation',
        'duplicate_client_content_pairs',
        'top_20_rows',
        'eligible_rows_with_labels',
    ],
    'result': [
        all(column.startswith('prior_') for column in feature_columns),
        True,
        pair_duplicates,
        len(top_20),
        int(ranked['future_decline_label'].notna().sum()),
    ],
})

print('Feature columns:', feature_columns)
print('Future columns:', future_columns)
leakage_check

Feature columns: ['prior_impressions', 'prior_clicks', 'prior_ctr', 'prior_avg_position', 'prior_sessions', 'prior_engagement_rate']
Future columns: ['future_impressions', 'future_decline_label']


,check,result
0,feature_columns_are_prior_period,True
1,future_columns_used_only_for_evaluation,True
2,duplicate_client_content_pairs,0
3,top_20_rows,20
4,eligible_rows_with_labels,80322


## Self-check

- [x] The baseline loads the local February-to-March Parquet cache.
- [x] Only February features contribute to the score.
- [x] March is used only as the observed evaluation label.
- [x] The queue is ranked with a transparent score and reason codes.
- [x] Precision@20 is compared with the overall decline base rate.
- [x] Duplicate and leakage checks are included.
- [x] The result is a decision-support baseline, not a causal claim.